<a href="https://colab.research.google.com/github/Adamphoenix003/GNN-LinkPrediction/blob/main/Heuristic_methods_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
uploaded = files.upload()



Saving cora.cites to cora.cites
Saving cora.content to cora.content


In [ ]:
edge_file = list(uploaded.keys())[0]
print("Loaded file:", edge_file)

Loaded file: cora.cites


In [ ]:
import networkx as nx

G = nx.Graph()

with open(edge_file, "r") as f:
    for line in f:
        u, v = line.strip().split()
        G.add_edge(u, v)

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

Nodes: 2708
Edges: 5278


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import random

edges = list(G.edges())
random.seed(42)
random.shuffle(edges)

num_test = int(0.1 * len(edges))

test_pos = edges[:num_test]
train_edges = edges[num_test:]

G_train = nx.Graph()
G_train.add_nodes_from(G.nodes())
G_train.add_edges_from(train_edges)

print("Train edges:", G_train.number_of_edges())

Train edges: 4751


In [ ]:
test_neg = set()

while len(test_neg) < num_test:
    u = random.choice(list(G.nodes()))
    v = random.choice(list(G.nodes()))
    if u != v and not G.has_edge(u, v):
        test_neg.add((u, v))

test_neg = list(test_neg)

test_edges = [(u, v, 1) for (u, v) in test_pos] + \
             [(u, v, 0) for (u, v) in test_neg]

print("Test samples:", len(test_edges))

Test samples: 1054


In [ ]:
import math

def cn_score(G, u, v):
    return len(list(nx.common_neighbors(G, u, v)))

def aa_score(G, u, v):
    score = 0
    for z in nx.common_neighbors(G, u, v):
        deg = G.degree(z)
        if deg > 1:
            score += 1 / math.log(deg)
    return score

def ra_score(G, u, v):
    score = 0
    for z in nx.common_neighbors(G, u, v):
        score += 1 / G.degree(z)
    return score

def sp_score(G, u, v):
    try:
        return 1 / nx.shortest_path_length(G, u, v)
    except nx.NetworkXNoPath:
        return 0

In [ ]:
from sklearn.metrics import average_precision_score, roc_auc_score

def evaluate(name, func):
    y_true = []
    y_score = []

    for u, v, label in test_edges:
        y_true.append(label)
        y_score.append(func(G_train, u, v))

    ap = average_precision_score(y_true, y_score)
    auc = roc_auc_score(y_true, y_score)

    print(f"{name}")
    print(f"  AP  : {ap:.4f}")
    print(f"  AUC : {auc:.4f}")
    print("-" * 30)

In [ ]:
evaluate("Common Neighbors (CN)", cn_score)
evaluate("Adamic-Adar (AA)", aa_score)
evaluate("Resource Allocation (RA)", ra_score)
evaluate("Shortest Path (SP)", sp_score)

Common Neighbors (CN)
  AP  : 0.7139
  AUC : 0.7163
------------------------------
Adamic-Adar (AA)
  AP  : 0.7195
  AUC : 0.7174
------------------------------
Resource Allocation (RA)
  AP  : 0.7194
  AUC : 0.7173
------------------------------
Shortest Path (SP)
  AP  : 0.8602
  AUC : 0.8168
------------------------------


In [ ]:
import numpy as np

nodes = list(G_train.nodes())
node_index = {n: i for i, n in enumerate(nodes)}

A = nx.to_numpy_array(G_train, nodelist=nodes)

beta = 0.005
L = 3

K = np.zeros_like(A)
A_power = A.copy()

for l in range(1, L + 1):
    K += (beta ** l) * A_power
    A_power = A_power @ A

def katz_score(u, v):
    return K[node_index[u], node_index[v]]

In [ ]:
y_true = []
y_score = []

for u, v, label in test_edges:
    y_true.append(label)
    y_score.append(katz_score(u, v))

from sklearn.metrics import average_precision_score, roc_auc_score

print("Katz")
print("  AP  :", round(average_precision_score(y_true, y_score), 4))
print("  AUC :", round(roc_auc_score(y_true, y_score), 4))

Katz
  AP  : 0.8142
  AUC : 0.8163
